# Modelo Principal — Denoising Autoencoder LSTM/GRU
## Teoría, Implementación Profunda y Análisis del Comportamiento del Modelo

**Proyecto:** Detección de Anomalías y Cambios de Régimen — ADRs Colombianos  
**Activos:** Ecopetrol (EC), Bancolombia (CIB), Grupo Aval (AVAL), Tecnoglass (TGLS)  
**Periodo:** 2015-01-01 — 2024-12-31  
**Autor:** [Nombre]  
**Versión:** 1.0

---

### Estructura del Notebook

| Sección | Contenido |
|---|---|
| §1 | Configuración y pipeline |
| §2 | Fundamento teórico del Autoencoder |
| §3 | Del Autoencoder estándar al Denoising Autoencoder |
| §4 | Arquitectura recurrente: LSTM y GRU en detalle |
| §5 | Implementación completa en TensorFlow/Keras |
| §6 | Implementación alternativa en PyTorch |
| §7 | Ruido gaussiano: teoría e implementación |
| §8 | Dropout: función regularizadora en series temporales |
| §9 | Error de reconstrucción MSE: definición y propiedades |
| §10 | Cómo el modelo aprende el comportamiento normal |
| §11 | Cómo se detectan las anomalías |
| §12 | Visualización profunda del error de reconstrucción |
| §13 | Análisis por feature: ¿qué reconstruye mal durante anomalías? |

---

> **Posición en la serie de notebooks:**  
> Este notebook profundiza en la teoría e implementación del modelo
> desarrollado en el Notebook 3. Mientras el NB3 establece el pipeline
> de entrenamiento y evaluación, este notebook justifica cada decisión
> de diseño desde primeros principios y provee visualizaciones detalladas
> del comportamiento interno del modelo.

---
## Configuración y Pipeline

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, math
import numpy as np
import pandas as pd
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score
)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import yfinance as yf

# TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Reproducibilidad global
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 11,
                     'axes.labelsize': 10})

print(f"TensorFlow : {tf.__version__}")
print(f"PyTorch    : {torch.__version__}")
print(f"GPU (TF)   : {tf.config.list_physical_devices('GPU')}")
print(f"GPU (PT)   : {'disponible' if torch.cuda.is_available() else 'no disponible'}")


In [ ]:
# ── Parámetros globales ───────────────────────────────────────────────────────
TICKERS      = ['EC', 'CIB', 'AVAL', 'TGLS']
NAMES        = {'EC': 'Ecopetrol', 'CIB': 'Bancolombia',
                'AVAL': 'Grupo Aval', 'TGLS': 'Tecnoglass'}
COLORS       = {'EC': '#1f77b4', 'CIB': '#ff7f0e',
                'AVAL': '#2ca02c', 'TGLS': '#d62728'}

TRAIN_START  = '2015-01-01';  TRAIN_END  = '2019-12-31'
VAL_START    = '2020-01-01';  VAL_END    = '2020-12-31'
TEST_START   = '2021-01-01';  TEST_END   = '2024-12-31'

SEQ_LEN      = 30
N_FEATURES   = 3
FEATURE_COLS = ['log_return', 'vol_21d', 'vol_zscore']
ROLL_WIN     = 21

# Hiperparámetros del DAE
LSTM_UNITS   = [64, 32]
LATENT_DIM   = 16
DROPOUT_RATE = 0.20
NOISE_STDDEV = 0.05
BATCH_SIZE   = 64
MAX_EPOCHS   = 150
LR           = 1e-3
PATIENCE_ES  = 15
PATIENCE_LR  = 7

ANOMALY_PERIODS = [
    {'name': 'Crisis petróleo', 'start': '2015-07-01',
     'end': '2016-02-01', 'color': 'rgba(128,0,128,0.10)'},
    {'name': 'COVID-19',        'start': '2020-02-15',
     'end': '2020-05-01', 'color': 'rgba(220,20,60,0.10)'},
    {'name': 'Fed hikes',       'start': '2022-01-01',
     'end': '2022-12-31', 'color': 'rgba(255,140,0,0.10)'},
]

TICKER_PILOT = 'EC'
print("Parámetros cargados.")


In [ ]:
# ── Pipeline de datos (idéntico a NB2) ───────────────────────────────────────
def build_features(ticker):
    raw = yf.download(ticker, start=TRAIN_START, end=TEST_END,
                      auto_adjust=True, progress=False)
    d = raw[['Open','High','Low','Close','Volume']].copy().dropna()
    d.index = pd.to_datetime(d.index)
    d['log_return'] = np.log(d['Close'] / d['Close'].shift(1))
    d['vol_21d']    = d['log_return'].rolling(ROLL_WIN).std() * np.sqrt(252)
    lv = np.log1p(d['Volume'])
    rs = lv.rolling(ROLL_WIN).std().replace(0, np.nan)
    d['vol_zscore'] = (lv - lv.rolling(ROLL_WIN).mean()) / rs
    return d[FEATURE_COLS].dropna()

def create_windows(arr, seq_len):
    return np.lib.stride_tricks.sliding_window_view(
        arr, window_shape=(seq_len, arr.shape[1])
    ).squeeze(1).copy()

def make_labels(dates_idx, anomaly_periods):
    labels = pd.Series(0, index=dates_idx)
    for ap in anomaly_periods:
        mask = (labels.index >= ap['start']) & (labels.index <= ap['end'])
        labels[mask] = 1
    return labels.values

def prepare(ticker):
    feat_df = build_features(ticker)
    splits_raw = {
        'train': feat_df.loc[:TRAIN_END],
        'val':   feat_df.loc[VAL_START:VAL_END],
        'test':  feat_df.loc[TEST_START:TEST_END],
    }
    scaler = StandardScaler()
    scaler.fit(splits_raw['train'].values)
    sc = {s: scaler.transform(df.values) for s, df in splits_raw.items()}
    wins, dates, labels = {}, {}, {}
    for s, arr in sc.items():
        w = create_windows(arr, SEQ_LEN)
        wins[s]   = w
        dates[s]  = splits_raw[s].index[SEQ_LEN - 1:]
        labels[s] = make_labels(dates[s], ANOMALY_PERIODS)
    return {'raw': splits_raw, 'sc': sc, 'wins': wins,
            'scaler': scaler, 'dates': dates, 'y': labels}

print("Construyendo pipeline...")
data = {t: prepare(t) for t in TICKERS}
d    = data[TICKER_PILOT]
print("Listo.")
print(f"  Train: {d['wins']['train'].shape}  "
      f"Val: {d['wins']['val'].shape}  "
      f"Test: {d['wins']['test'].shape}")


---
## Fundamento Teórico del Autoencoder

### **2.1 Definición formal**

Un autoencoder es un modelo generativo que aprende una función de identidad
restringida. Formalmente, es una red neuronal que aproxima la transformación:

```
x̂ = f(x) ≈ x,    con la restricción dim(z) << dim(x)
```

La restricción dimensional se implementa mediante dos componentes:

**Encoder:** función parametrizada fθ que comprime la entrada al espacio latente:

```
z = fθ(x),    x ∈ R^n,  z ∈ R^d,  d << n
```

**Decoder:** función parametrizada gφ que reconstruye la entrada desde el espacio latente:

```
x̂ = gφ(z) = gφ(fθ(x))
```

**Función de pérdida:** error de reconstrucción cuadrático medio:

```
L(θ, φ) = (1/N) Σᵢ ||xᵢ - gφ(fθ(xᵢ))||²
         = E[||x - x̂||²]
```

### **2.2 El principio de compresión selectiva**

La clave del autoencoder como detector de anomalías reside en la **selectividad
de la compresión**. El cuello de botella (dimensión d << n) fuerza al modelo a
aprender sólo las direcciones del espacio de entrada que concentran la mayor
varianza de los datos de entrenamiento.

**Geometría del aprendizaje:**

```
Espacio de entrada R^n
        │
        │ fθ proyecta sobre
        │ el manifold aprendido
        ▼
Manifold de datos normales M ⊂ R^n
        │
        │ Datos normales: z = fθ(x) ∈ Z_normal  (región densa del espacio latente)
        │ Datos anómalos: z' = fθ(x') ∉ Z_normal (región escasamente vista)
        ▼
Espacio latente R^d
```

Cuando el decoder intenta reconstruir desde una representación z' fuera de la
región normal, produce x̂' alejado de x', generando alto error de reconstrucción.

### **2.3 Analogía financiera**

El autoencoder aprende la "gramática" del mercado normal: qué combinaciones
de retorno, volatilidad y volumen son coherentes en condiciones típicas.
Durante una crisis, el mercado "habla un idioma diferente" — los patrones
violan la gramática aprendida y el modelo falla en reconstruirlos.

In [ ]:
# ── Visualización conceptual: manifold normal vs anómalo ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── Panel 1: Distribución de datos en 2D (simulada) ──────────────────────────
np.random.seed(SEED)
n_normal  = 800
n_anomaly = 60

# Datos normales: distribución elíptica compacta
X_norm_vis = np.random.multivariate_normal(
    mean=[0, 0],
    cov=[[1, 0.6], [0.6, 1]],
    size=n_normal
)
# Datos anómalos: dispersos en la periferia
X_anom_vis = np.random.multivariate_normal(
    mean=[3.5, -3.0],
    cov=[[1.5, 0], [0, 1.5]],
    size=n_anomaly
)

axes[0].scatter(X_norm_vis[:, 0], X_norm_vis[:, 1],
                c='#1f77b4', s=12, alpha=0.4, label='Normal (train)')
axes[0].scatter(X_anom_vis[:, 0], X_anom_vis[:, 1],
                c='#d62728', s=30, alpha=0.8, marker='^', label='Anómalo (test)')

# Elipse del manifold normal (aprox. 2σ)
theta  = np.linspace(0, 2 * np.pi, 200)
a, b   = 2.3, 1.4
angle  = np.pi / 4
x_ell  = a * np.cos(theta) * np.cos(angle) - b * np.sin(theta) * np.sin(angle)
y_ell  = a * np.cos(theta) * np.sin(angle) + b * np.sin(theta) * np.cos(angle)
axes[0].plot(x_ell, y_ell, 'g--', linewidth=1.8, label='Manifold normal (2σ)')
axes[0].set_title('1. Distribución de datos
(espacio de entrada)',
                  fontweight='bold')
axes[0].set_xlabel('Feature 1'); axes[0].set_ylabel('Feature 2')
axes[0].legend(fontsize=8)

# ── Panel 2: Proyección al espacio latente ────────────────────────────────────
# El encoder proyecta todo al espacio latente
# Datos normales → región densa; anómalos → región dispersa
z_normal  = np.random.normal(0, 0.5, (n_normal, 2))
z_anomaly = np.random.normal(3, 0.8, (n_anomaly, 2))

axes[1].scatter(z_normal[:, 0],  z_normal[:, 1],
                c='#1f77b4', s=12, alpha=0.4, label='Normal')
axes[1].scatter(z_anomaly[:, 0], z_anomaly[:, 1],
                c='#d62728', s=30, alpha=0.8, marker='^', label='Anómalo')
circle = plt.Circle((0, 0), 1.8, fill=False, color='green',
                     linestyle='--', linewidth=1.8, label='Región latente normal')
axes[1].add_patch(circle)
axes[1].set_xlim(-4, 6); axes[1].set_ylim(-4, 6)
axes[1].set_aspect('equal')
axes[1].set_title('2. Espacio latente z ∈ R^d
(después del encoder)',
                  fontweight='bold')
axes[1].set_xlabel('z₁'); axes[1].set_ylabel('z₂')
axes[1].legend(fontsize=8)

# ── Panel 3: Error de reconstrucción ─────────────────────────────────────────
# El error de reconstrucción es bajo para normales y alto para anómalos
mse_normal  = np.random.exponential(0.02, n_normal)
mse_anomaly = np.random.exponential(0.25, n_anomaly) + 0.10
tau_vis     = np.percentile(mse_normal, 95)

axes[2].hist(mse_normal,  bins=50, density=True, color='#1f77b4',
             alpha=0.6, label='Normal (train)', edgecolor='none')
axes[2].hist(mse_anomaly, bins=25, density=True, color='#d62728',
             alpha=0.6, label='Anómalo', edgecolor='none')
axes[2].axvline(tau_vis, color='green', linewidth=2.0, linestyle='--',
                label=f'Umbral τ (p95 train) = {tau_vis:.3f}')
axes[2].set_title('3. Distribución del MSE
(score de anomalía)',
                  fontweight='bold')
axes[2].set_xlabel('MSE de reconstrucción')
axes[2].set_ylabel('Densidad')
axes[2].legend(fontsize=8)

plt.suptitle('Principio del Autoencoder como Detector de Anomalías',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_autoencoder_concept.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Del Autoencoder Estándar al Denoising Autoencoder

### **3.1 Limitación del autoencoder estándar**

Un autoencoder estándar entrenado con suficiente capacidad puede
aprender la función identidad trivial: memorizar cada muestra individual
en el espacio latente. Este sobreajuste elimina la separabilidad entre
datos normales y anómalos, porque el modelo aprende a reconstruir
perfectamente cualquier entrada, incluyendo las anómalas.

Formalmente, el riesgo es que el encoder aprenda una inyección
(función uno-a-uno) en lugar de una proyección sobre el manifold normal.

### **3.2 El Denoising Autoencoder (DAE)**

Vincent et al. (2008) proponen una solución elegante: corromper la entrada
con ruido aleatorio durante el entrenamiento, mientras el objetivo de
reconstrucción permanece siendo la entrada original limpia.

**Función objetivo del DAE:**

```
x̃ = q(x̃|x)    ← proceso de corrupción estocástico
L_DAE = E_{x,x̃} [||x - gφ(fθ(x̃))||²]
```

Para ruido gaussiano aditivo:

```
x̃ = x + ε,    ε ~ N(0, σ²_noise · I)
L_DAE = (1/N) Σᵢ ||xᵢ - gφ(fθ(xᵢ + εᵢ))||²
```

### **3.3 Por qué el ruido previene la memorización**

**Intuición geométrica:** el ruido fuerza al modelo a aprender una
función de promediado local alrededor de cada punto de entrenamiento.
En lugar de memorizar cada x, el encoder debe aprender a mapear
toda la vecindad ruidosa de x a la misma representación latente z.
Esto equivale a aprender una proyección robusta sobre el manifold
de los datos normales.

**Efecto sobre la detección de anomalías:**

```
Datos normales:  xᵢ + ε → zᵢ ∈ Z_normal → reconstrucción fiel  → MSE bajo
Datos anómalos:  x'ᵢ + ε → z'ᵢ ∉ Z_normal → reconstrucción pobre → MSE alto
```

El ruido desplaza cada punto de entrenamiento en direcciones aleatorias,
cubriendo implícitamente una región más amplia del espacio de entrada
alrededor del manifold normal. El decoder aprende a reconstruir desde
cualquier punto de esta región, pero sólo desde ella.

### **3.4 Elección de σ_noise**

```
σ_noise << σ_features  → ruido insuficiente, memorización posible
σ_noise ≈ σ_features   → ruido excesivo, señal destruida
σ_noise ≈ 0.05 × σ_features → equilibrio óptimo para retornos financieros
```

Con features escaladas a σ = 1, se usa σ_noise = 0.05.

In [ ]:
# ── Visualización: AE estándar vs DAE — sensibilidad al ruido ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

noise_levels = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.50]

# Simular cómo cambia el MSE de reconstrucción para datos normales y anómalos
# en función del nivel de ruido de entrenamiento (resultado conceptual)
np.random.seed(SEED)
mse_normal_sim  = [0.008, 0.009, 0.010, 0.013, 0.018, 0.030, 0.070]
mse_anomaly_sim = [0.015, 0.025, 0.040, 0.090, 0.110, 0.095, 0.075]
separability    = [m_a - m_n for m_n, m_a
                   in zip(mse_normal_sim, mse_anomaly_sim)]

axes[0].plot(noise_levels, mse_normal_sim,  'o-',
             color='#1f77b4', linewidth=1.8, label='MSE — Normal')
axes[0].plot(noise_levels, mse_anomaly_sim, 's-',
             color='#d62728', linewidth=1.8, label='MSE — Anómalo')
axes[0].axvline(0.05, color='green', linewidth=1.5, linestyle='--',
                label='σ_noise = 0.05 (seleccionado)')
axes[0].fill_between(noise_levels, mse_normal_sim, mse_anomaly_sim,
                     alpha=0.10, color='green')
axes[0].set_xlabel('σ_noise (nivel de ruido de entrenamiento)')
axes[0].set_ylabel('MSE de reconstrucción promedio')
axes[0].set_title('MSE de Normal vs Anómalo
en función del ruido',
                  fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].plot(noise_levels, separability, 'D-',
             color='#9467bd', linewidth=2.0)
axes[1].axvline(0.05, color='green', linewidth=1.5, linestyle='--',
                label='σ_noise = 0.05 (max separabilidad)')
axes[1].fill_between(noise_levels, separability,
                     alpha=0.15, color='#9467bd')
axes[1].set_xlabel('σ_noise')
axes[1].set_ylabel('MSE_anómalo - MSE_normal (separabilidad)')
axes[1].set_title('Separabilidad de regímenes
vs nivel de ruido',
                  fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Denoising Autoencoder: Efecto del Nivel de Ruido sobre la Detección',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_noise_level_effect.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Arquitectura Recurrente: LSTM y GRU en Detalle

### **4.1 Motivación para celdas recurrentes**

El vector de features en cada timestep x_t ∈ R^F no es independiente de
x_{t-1}: la volatilidad del día de hoy está correlacionada con la de ayer
(efecto ARCH, confirmado en EDA §6). Un encoder basado en capas densas
ignora esta estructura temporal y procesa cada timestep de forma independiente.

Las celdas recurrentes (LSTM, GRU) mantienen un **estado oculto** h_t que
actúa como memoria de las observaciones anteriores, permitiendo al encoder
comprimir no sólo el valor puntual de cada feature sino también su
evolución temporal.

### **4.2 Long Short-Term Memory (LSTM)**

El LSTM introduce tres compuertas que controlan el flujo de información
a través del tiempo:

```
Compuerta de olvido (forget gate):
  f_t = σ(W_f · [h_{t-1}, x_t] + b_f)
  → Decide qué información del estado de celda anterior descartar

Compuerta de entrada (input gate):
  i_t = σ(W_i · [h_{t-1}, x_t] + b_i)
  C̃_t = tanh(W_C · [h_{t-1}, x_t] + b_C)
  → Decide qué información nueva añadir al estado de celda

Actualización del estado de celda:
  C_t = f_t ⊙ C_{t-1} + i_t ⊙ C̃_t
  → Combina memoria antigua (filtrada) con información nueva

Compuerta de salida (output gate):
  o_t = σ(W_o · [h_{t-1}, x_t] + b_o)
  h_t = o_t ⊙ tanh(C_t)
  → Produce el estado oculto que se pasa al siguiente timestep
```

**Número de parámetros por capa LSTM con `units` u e input de dimensión d:**
```
4 × (u × (u + d) + u) = 4u(u + d + 1)
```

### **4.3 Gated Recurrent Unit (GRU)**

El GRU simplifica el LSTM fusionando las compuertas de olvido y entrada:

```
Compuerta de actualización (update gate):
  z_t = σ(W_z · [h_{t-1}, x_t])
  → Controla cuánto del estado anterior se conserva

Compuerta de reinicio (reset gate):
  r_t = σ(W_r · [h_{t-1}, x_t])
  → Controla qué del estado anterior se usa para calcular h̃_t

Estado candidato:
  h̃_t = tanh(W · [r_t ⊙ h_{t-1}, x_t])

Actualización del estado oculto:
  h_t = (1 - z_t) ⊙ h_{t-1} + z_t ⊙ h̃_t
  → Interpola entre el estado anterior y el candidato
```

**Número de parámetros por capa GRU:**
```
3 × (u × (u + d) + u) = 3u(u + d + 1)   ← ~25% menos que LSTM
```

### **4.4 Arquitectura completa del DAE**

```
╔══════════════════════════════════════════════════════════╗
║              DENOISING AUTOENCODER LSTM/GRU              ║
╠══════════════════════════════════════════════════════════╣
║  INPUT: (B, T=30, F=3)  ← ventana ruidosa x̃            ║
╠══════════════════════════════════════════════════════════╣
║  ENCODER                                                 ║
║  ┌────────────────────────────────────────────────────┐  ║
║  │ LSTM/GRU(64, return_sequences=True)                │  ║
║  │   → (B, 30, 64)  ← secuencia de estados ocultos   │  ║
║  │ Dropout(0.20)                                      │  ║
║  │ LSTM/GRU(32, return_sequences=False)               │  ║
║  │   → (B, 32)      ← estado final h_T               │  ║
║  │ Dropout(0.20)                                      │  ║
║  │ Dense(16, tanh)  → z ∈ R^16  ← espacio latente    │  ║
║  └────────────────────────────────────────────────────┘  ║
╠══════════════════════════════════════════════════════════╣
║  CUELLO DE BOTELLA: z ∈ R^16                             ║
║  Ratio de compresión: 30×3 / 16 = 5.6×                  ║
╠══════════════════════════════════════════════════════════╣
║  DECODER                                                 ║
║  ┌────────────────────────────────────────────────────┐  ║
║  │ RepeatVector(30)  → (B, 30, 16)                    │  ║
║  │ LSTM/GRU(32, return_sequences=True)                │  ║
║  │   → (B, 30, 32)                                    │  ║
║  │ Dropout(0.20)                                      │  ║
║  │ LSTM/GRU(64, return_sequences=True)                │  ║
║  │   → (B, 30, 64)                                    │  ║
║  │ Dropout(0.20)                                      │  ║
║  │ TimeDistributed(Dense(3))                          │  ║
║  │   → (B, 30, 3)   ← reconstrucción x̂              │  ║
║  └────────────────────────────────────────────────────┘  ║
╠══════════════════════════════════════════════════════════╣
║  OUTPUT: (B, T=30, F=3)  ← reconstrucción x̂            ║
║  LOSS: MSE(x_limpia, x̂)  ← siempre sobre entrada limpia║
╚══════════════════════════════════════════════════════════╝
```

In [ ]:
# ── Diagrama de parámetros por capa ──────────────────────────────────────────
def count_lstm_params(units, input_dim):
    return 4 * units * (units + input_dim + 1)

def count_gru_params(units, input_dim):
    return 3 * units * (units + input_dim + 1)

print("ANÁLISIS DE PARÁMETROS — DAE LSTM vs GRU")
print("=" * 65)

layers_config = [
    # (nombre, tipo, units, input_dim)
    ('Encoder LSTM/GRU 1', 64, N_FEATURES),
    ('Encoder LSTM/GRU 2', 32, 64),
    ('Dense latente',      None, None),
    ('Decoder LSTM/GRU 1', 32, LATENT_DIM),
    ('Decoder LSTM/GRU 2', 64, 32),
    ('TimeDistributed',    None, None),
]

total_lstm = total_gru = 0
print(f"{'Capa':<25}  {'LSTM params':>14}  {'GRU params':>14}  {'Diferencia':>12}")
print('-' * 68)

for name, units, inp_dim in layers_config:
    if units is None:
        # Dense latente: 32 → 16
        p = 32 * LATENT_DIM + LATENT_DIM
        print(f"{name:<25}  {p:>14,}  {p:>14,}  {'—':>12}")
        total_lstm += p; total_gru += p
    elif name.startswith('Time'):
        p = 64 * N_FEATURES + N_FEATURES   # TimeDistributed(Dense(3))
        print(f"{name:<25}  {p:>14,}  {p:>14,}  {'—':>12}")
        total_lstm += p; total_gru += p
    else:
        pl = count_lstm_params(units, inp_dim)
        pg = count_gru_params(units, inp_dim)
        diff = pl - pg
        print(f"{name:<25}  {pl:>14,}  {pg:>14,}  {diff:>12,}")
        total_lstm += pl; total_gru += pg

print('-' * 68)
print(f"{'TOTAL':<25}  {total_lstm:>14,}  {total_gru:>14,}  "
      f"{total_lstm - total_gru:>12,}")
print()
print(f"Reducción GRU vs LSTM: {(1 - total_gru/total_lstm)*100:.1f}%")
print(f"Ratio de compresión  : {SEQ_LEN * N_FEATURES} → {LATENT_DIM} "
      f"= {SEQ_LEN * N_FEATURES / LATENT_DIM:.1f}×")


---
## Implementación Completa en TensorFlow/Keras

### **5.1 Diseño modular**

La implementación se estructura en tres clases/funciones independientes:
- `GaussianNoise`: capa personalizada de ruido (sólo activa en entrenamiento)
- `build_encoder`: construye el sub-modelo encoder como `keras.Model`
- `build_dae_tf`: construye el DAE completo
- `DenoisingAutoencoder`: clase wrapper con métodos `fit`, `score`, `detect`

In [ ]:
# ── Capa de ruido gaussiano personalizada ────────────────────────────────────
class GaussianNoiseLayer(layers.Layer):
    """
    Añade ruido gaussiano aditivo a la entrada.
    El ruido se aplica SÓLO durante el entrenamiento (training=True).
    Durante la inferencia la entrada pasa sin modificación.

    Esto garantiza que el score de anomalía en inferencia refleje
    únicamente la dificultad real de reconstrucción, sin ruido artificial.

    Parámetros
    ----------
    stddev : float — desviación estándar del ruido gaussiano
    """
    def __init__(self, stddev, **kwargs):
        super().__init__(**kwargs)
        self.stddev = stddev

    def call(self, inputs, training=None):
        if training:
            noise = tf.random.normal(shape=tf.shape(inputs),
                                     mean=0.0, stddev=self.stddev)
            return inputs + noise
        return inputs   # inferencia: sin ruido

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'stddev': self.stddev})
        return cfg


# ── Constructor del encoder como sub-modelo independiente ────────────────────
def build_encoder_tf(seq_len, n_feat, latent_dim, lstm_units,
                     dropout_rate, cell_type='LSTM'):
    """
    Construye el encoder como keras.Model independiente.
    Permite extraer representaciones latentes z sin necesidad
    de ejecutar el decoder.
    """
    RNN = layers.LSTM if cell_type == 'LSTM' else layers.GRU

    inp = keras.Input(shape=(seq_len, n_feat), name='enc_input')
    x   = RNN(lstm_units[0], return_sequences=True,
              name='enc_rnn_1')(inp)
    x   = layers.Dropout(dropout_rate, name='enc_drop_1')(x)
    x   = RNN(lstm_units[1], return_sequences=False,
              name='enc_rnn_2')(x)
    x   = layers.Dropout(dropout_rate, name='enc_drop_2')(x)
    z   = layers.Dense(latent_dim, activation='tanh',
                       name='latent_z')(x)

    return Model(inp, z, name=f'Encoder_{cell_type}')


# ── Constructor del DAE completo ──────────────────────────────────────────────
def build_dae_tf(seq_len, n_feat, latent_dim, lstm_units,
                 dropout_rate, noise_stddev, cell_type='LSTM'):
    """
    Construye el Denoising Autoencoder completo.

    Arquitectura:
      GaussianNoiseLayer → Encoder LSTM/GRU → [z] → Decoder LSTM/GRU

    El ruido se aplica como primera capa del modelo, dentro del grafo
    computacional de Keras, garantizando que sea automáticamente inactivo
    durante la inferencia (model.predict / model.evaluate).

    Parámetros
    ----------
    seq_len       : int — longitud de secuencia (T)
    n_feat        : int — número de features (F)
    latent_dim    : int — dimensión del espacio latente
    lstm_units    : list[int] — [units_1, units_2] del encoder
                   El decoder usa el orden inverso.
    dropout_rate  : float — tasa de dropout
    noise_stddev  : float — σ del ruido gaussiano (sólo en training)
    cell_type     : 'LSTM' o 'GRU'

    Retorna
    -------
    model         : keras.Model compilado
    encoder       : keras.Model del sub-encoder (para análisis latente)
    """
    RNN          = layers.LSTM if cell_type == 'LSTM' else layers.GRU
    dec_units    = lstm_units[::-1]   # [32, 64]
    model_name   = f'DAE_{cell_type}'

    # ── Entrada y ruido ───────────────────────────────────────────────────────
    inp    = keras.Input(shape=(seq_len, n_feat), name='input')
    noisy  = GaussianNoiseLayer(noise_stddev, name='gaussian_noise')(inp)

    # ── Encoder ──────────────────────────────────────────────────────────────
    x      = RNN(lstm_units[0], return_sequences=True,
                 name='enc_rnn_1')(noisy)
    x      = layers.Dropout(dropout_rate, name='enc_drop_1')(x)
    x      = RNN(lstm_units[1], return_sequences=False,
                 name='enc_rnn_2')(x)
    x      = layers.Dropout(dropout_rate, name='enc_drop_2')(x)
    z      = layers.Dense(latent_dim, activation='tanh',
                          name='latent_z')(x)

    # ── Decoder ──────────────────────────────────────────────────────────────
    y      = layers.RepeatVector(seq_len, name='repeat_z')(z)
    y      = RNN(dec_units[0], return_sequences=True,
                 name='dec_rnn_1')(y)
    y      = layers.Dropout(dropout_rate, name='dec_drop_1')(y)
    y      = RNN(dec_units[1], return_sequences=True,
                 name='dec_rnn_2')(y)
    y      = layers.Dropout(dropout_rate, name='dec_drop_2')(y)
    output = layers.TimeDistributed(
        layers.Dense(n_feat), name='output_proj'
    )(y)

    # ── Modelos ───────────────────────────────────────────────────────────────
    dae_model = Model(inp, output, name=model_name)
    dae_model.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss='mse'
    )

    # Sub-encoder: del input al espacio latente (sin ruido — para inferencia)
    # NOTA: el encoder de inferencia NO pasa por GaussianNoiseLayer
    enc_input = keras.Input(shape=(seq_len, n_feat), name='enc_only_input')
    ex   = RNN(lstm_units[0], return_sequences=True,  name='enc_rnn_1')(enc_input)
    ex   = layers.Dropout(dropout_rate, name='enc_drop_1')(ex)
    ex   = RNN(lstm_units[1], return_sequences=False, name='enc_rnn_2')(ex)
    ex   = layers.Dropout(dropout_rate, name='enc_drop_2')(ex)
    ez   = layers.Dense(latent_dim, activation='tanh', name='latent_z')(ex)
    encoder = Model(enc_input, ez, name=f'Encoder_{cell_type}')

    return dae_model, encoder


# ── Clase wrapper ─────────────────────────────────────────────────────────────
class DenoisingAutoencoderTF:
    """
    Wrapper del DAE con interfaz simplificada para:
      - Entrenamiento con callbacks estándar
      - Cómputo del score de anomalía (MSE por ventana)
      - Detección de anomalías con umbral calibrado
      - Extracción de representaciones latentes
    """
    def __init__(self, seq_len, n_feat, latent_dim, lstm_units,
                 dropout_rate, noise_stddev, cell_type='LSTM'):
        self.model, self.encoder = build_dae_tf(
            seq_len, n_feat, latent_dim, lstm_units,
            dropout_rate, noise_stddev, cell_type
        )
        self.tau         = None
        self.history     = None
        self.cell_type   = cell_type

    def fit(self, X_train, X_val, batch_size=64,
            max_epochs=150, patience_es=15, patience_lr=7):
        """
        Entrena el DAE.
        Input y target son idénticos (X_train) — el ruido se añade
        internamente por GaussianNoiseLayer durante training=True.
        """
        cb_list = [
            callbacks.EarlyStopping(
                monitor='val_loss', patience=patience_es,
                restore_best_weights=True, verbose=0
            ),
            callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5,
                patience=patience_lr, min_lr=1e-6, verbose=0
            ),
        ]
        self.history = self.model.fit(
            X_train.astype(np.float32),
            X_train.astype(np.float32),   # target = entrada limpia
            validation_data=(
                X_val.astype(np.float32),
                X_val.astype(np.float32)
            ),
            batch_size=batch_size,
            epochs=max_epochs,
            callbacks=cb_list,
            shuffle=True,
            verbose=0
        )
        return self

    def score(self, X):
        """Retorna el MSE de reconstrucción por ventana. Shape: (n,)."""
        Xf   = X.astype(np.float32)
        Xhat = self.model.predict(Xf, batch_size=256, verbose=0)
        return np.mean((Xf - Xhat) ** 2, axis=(1, 2))

    def set_threshold(self, X_train, percentile=95):
        """Calibra el umbral τ sobre el conjunto de entrenamiento."""
        scores    = self.score(X_train)
        self.tau  = np.percentile(scores, percentile)
        return self.tau

    def detect(self, X):
        """Retorna etiquetas binarias: 1=anomalía, 0=normal."""
        if self.tau is None:
            raise ValueError("Llame set_threshold() antes de detect().")
        return (self.score(X) > self.tau).astype(int)

    def encode(self, X):
        """Extrae representaciones latentes z. Shape: (n, latent_dim)."""
        # Copiar pesos del encoder entrenado al sub-encoder de inferencia
        for layer in self.model.layers:
            if layer.name in [l.name for l in self.encoder.layers]:
                try:
                    self.encoder.get_layer(layer.name).set_weights(
                        layer.get_weights()
                    )
                except Exception:
                    pass
        return self.encoder.predict(X.astype(np.float32),
                                    batch_size=256, verbose=0)

    def summary(self):
        self.model.summary()


# Instanciar y mostrar resumen
dae_tf = DenoisingAutoencoderTF(
    seq_len=SEQ_LEN, n_feat=N_FEATURES,
    latent_dim=LATENT_DIM, lstm_units=LSTM_UNITS,
    dropout_rate=DROPOUT_RATE, noise_stddev=NOISE_STDDEV,
    cell_type='LSTM'
)
dae_tf.summary()


---
## Implementación Alternativa en PyTorch

### **Motivación de la implementación dual**

La implementación en PyTorch permite:
1. Mayor control sobre el loop de entrenamiento (útil para variantes avanzadas).
2. Integración con herramientas de interpretabilidad (Captum, GradCAM).
3. Comparación directa de las dos implementaciones como validación cruzada.

### **Diferencias clave respecto a Keras**

| Aspecto | TensorFlow/Keras | PyTorch |
|---|---|---|
| Ruido | `GaussianNoiseLayer` integrada | Aplicado explícitamente en el loop |
| Dropout | Automático (train/eval mode) | Requiere `model.train()` / `model.eval()` |
| Forward pass | Declarativo (grafo) | Imperativo (eager por defecto) |
| Ventaja | Menor código boilerplate | Mayor flexibilidad y control |

In [ ]:
# ── Implementación PyTorch del DAE ───────────────────────────────────────────
class DAE_PyTorch(nn.Module):
    """
    Denoising Autoencoder recurrente implementado en PyTorch.
    Soporta celdas LSTM y GRU.

    Parámetros
    ----------
    seq_len      : int — longitud de secuencia T
    n_features   : int — número de features F
    latent_dim   : int — dimensión del espacio latente
    hidden_sizes : list[int] — unidades por capa del encoder [u1, u2]
    dropout      : float — tasa de dropout
    cell_type    : str — 'LSTM' o 'GRU'
    """
    def __init__(self, seq_len, n_features, latent_dim,
                 hidden_sizes, dropout, cell_type='LSTM'):
        super().__init__()
        self.seq_len    = seq_len
        self.n_features = n_features
        self.latent_dim = latent_dim
        self.cell_type  = cell_type

        RNNClass = nn.LSTM if cell_type == 'LSTM' else nn.GRU

        # ── Encoder ──────────────────────────────────────────────────────────
        self.enc_rnn1 = RNNClass(
            input_size=n_features,
            hidden_size=hidden_sizes[0],
            num_layers=1,
            batch_first=True
        )
        self.enc_drop1 = nn.Dropout(dropout)

        self.enc_rnn2 = RNNClass(
            input_size=hidden_sizes[0],
            hidden_size=hidden_sizes[1],
            num_layers=1,
            batch_first=True
        )
        self.enc_drop2 = nn.Dropout(dropout)

        # Capa densa al espacio latente
        self.enc_dense = nn.Sequential(
            nn.Linear(hidden_sizes[1], latent_dim),
            nn.Tanh()
        )

        # ── Decoder ──────────────────────────────────────────────────────────
        dec_sizes = hidden_sizes[::-1]   # [32, 64]

        self.dec_rnn1 = RNNClass(
            input_size=latent_dim,
            hidden_size=dec_sizes[0],
            num_layers=1,
            batch_first=True
        )
        self.dec_drop1 = nn.Dropout(dropout)

        self.dec_rnn2 = RNNClass(
            input_size=dec_sizes[0],
            hidden_size=dec_sizes[1],
            num_layers=1,
            batch_first=True
        )
        self.dec_drop2 = nn.Dropout(dropout)

        # Proyección de salida (equivalente a TimeDistributed(Dense))
        self.output_proj = nn.Linear(dec_sizes[1], n_features)

    def encode(self, x):
        """Codifica una secuencia al espacio latente. x: (B, T, F)"""
        out1, _ = self.enc_rnn1(x)           # (B, T, h1)
        out1     = self.enc_drop1(out1)
        out2, _  = self.enc_rnn2(out1)       # (B, T, h2)
        out2     = self.enc_drop2(out2)
        h_final  = out2[:, -1, :]             # (B, h2) — último timestep
        z        = self.enc_dense(h_final)    # (B, latent_dim)
        return z

    def decode(self, z):
        """Decodifica z al espacio de secuencias. z: (B, latent_dim)"""
        # RepeatVector: replica z para cada timestep
        z_rep  = z.unsqueeze(1).repeat(1, self.seq_len, 1)  # (B, T, latent_dim)
        out1, _ = self.dec_rnn1(z_rep)                       # (B, T, d1)
        out1    = self.dec_drop1(out1)
        out2, _ = self.dec_rnn2(out1)                        # (B, T, d2)
        out2    = self.dec_drop2(out2)
        recon   = self.output_proj(out2)                     # (B, T, F)
        return recon

    def forward(self, x):
        """Forward pass completo. x: (B, T, F)"""
        z     = self.encode(x)
        x_hat = self.decode(z)
        return x_hat


class DenoisingAutoencoderPT:
    """
    Wrapper de entrenamiento para DAE_PyTorch.
    Equivalente funcional a DenoisingAutoencoderTF.
    """
    def __init__(self, seq_len, n_feat, latent_dim, hidden_sizes,
                 dropout, noise_stddev, cell_type='LSTM',
                 lr=1e-3, device=None):
        self.noise_stddev = noise_stddev
        self.tau          = None
        self.history      = {'train_loss': [], 'val_loss': []}
        self.device       = device or (
            'cuda' if torch.cuda.is_available() else 'cpu'
        )
        self.model = DAE_PyTorch(
            seq_len, n_feat, latent_dim,
            hidden_sizes, dropout, cell_type
        ).to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        self.criterion = nn.MSELoss()

    def _add_noise(self, X_tensor):
        return X_tensor + torch.randn_like(X_tensor) * self.noise_stddev

    def fit(self, X_train_np, X_val_np, batch_size=64,
            max_epochs=150, patience=15):
        X_tr_t = torch.tensor(X_train_np, dtype=torch.float32)
        X_vl_t = torch.tensor(X_val_np,  dtype=torch.float32)
        loader  = DataLoader(TensorDataset(X_tr_t),
                             batch_size=batch_size, shuffle=True)

        best_val, patience_count, best_state = np.inf, 0, None

        for epoch in range(max_epochs):
            # ── Entrenamiento ────────────────────────────────────────────────
            self.model.train()
            epoch_loss = []
            for (Xb,) in loader:
                Xb      = Xb.to(self.device)
                Xb_noisy= self._add_noise(Xb)   # ruido sólo en training
                Xb_hat  = self.model(Xb_noisy)
                loss    = self.criterion(Xb_hat, Xb)   # target = limpio
                self.optimizer.zero_grad()
                loss.backward()
                # Gradient clipping para estabilidad del LSTM
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), max_norm=1.0
                )
                self.optimizer.step()
                epoch_loss.append(loss.item())

            # ── Validación ───────────────────────────────────────────────────
            self.model.eval()
            with torch.no_grad():
                X_vl_dev = X_vl_t.to(self.device)
                val_hat  = self.model(X_vl_dev)
                val_loss = self.criterion(val_hat, X_vl_dev).item()

            tr_loss = np.mean(epoch_loss)
            self.history['train_loss'].append(tr_loss)
            self.history['val_loss'].append(val_loss)

            # Early stopping
            if val_loss < best_val:
                best_val     = val_loss
                patience_count = 0
                best_state   = {k: v.clone()
                                for k, v in self.model.state_dict().items()}
            else:
                patience_count += 1
                if patience_count >= patience:
                    self.model.load_state_dict(best_state)
                    break

        return self

    def score(self, X_np):
        """MSE de reconstrucción por ventana. Sin ruido."""
        self.model.eval()
        X_t = torch.tensor(X_np, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            X_hat = self.model(X_t)
        mse = ((X_t - X_hat) ** 2).mean(dim=(1, 2)).cpu().numpy()
        return mse

    def set_threshold(self, X_train_np, percentile=95):
        self.tau = np.percentile(self.score(X_train_np), percentile)
        return self.tau

    def detect(self, X_np):
        return (self.score(X_np) > self.tau).astype(int)


# ── Demostración: entrenamiento del DAE en PyTorch ────────────────────────────
print("Instanciando y entrenando DAE-LSTM en PyTorch...")
dae_pt = DenoisingAutoencoderPT(
    seq_len=SEQ_LEN, n_feat=N_FEATURES,
    latent_dim=LATENT_DIM, hidden_sizes=LSTM_UNITS,
    dropout=DROPOUT_RATE, noise_stddev=NOISE_STDDEV,
    cell_type='LSTM', lr=LR
)
dae_pt.fit(
    d['wins']['train'], d['wins']['val'],
    batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
    patience=PATIENCE_ES
)
print(f"Entrenamiento completado.")
print(f"  Épocas ejecutadas : {len(dae_pt.history['train_loss'])}")
print(f"  Val loss mínima   : {min(dae_pt.history['val_loss']):.6f}")


---
## Ruido Gaussiano: Teoría e Implementación

### **7.1 Propiedades del ruido gaussiano aditivo**

El ruido gaussiano aditivo ε ~ N(0, σ²·I) tiene propiedades que lo hacen
idóneo para el DAE financiero:

- **Isotropía:** la perturbación es equidistribuida en todas las direcciones
  del espacio de features, sin sesgar ninguna feature en particular.
- **Continuidad:** las muestras ruidosas son vecinas de las originales en el
  espacio de features, por lo que el modelo aprende una función de denoising
  localmente consistente.
- **Controlabilidad:** un único parámetro σ_noise controla la intensidad
  de la perturbación, facilitando la búsqueda de hiperparámetros.

### **7.2 Equivalencia con regularización de la representación**

Puede demostrarse que, bajo ciertas condiciones, entrenar un DAE con
ruido gaussiano es equivalente a entrenar un autoencoder estándar
con regularización de la norma del Jacobiano del encoder:

```
L_DAE ≈ L_AE + λ · ||∂fθ/∂x||²_F
```

Esto explica por qué el DAE produce representaciones latentes más suaves
y robustas que el AE estándar: el ruido actúa como regularizador implícito
que penaliza los gradientes demasiado grandes del encoder.

In [ ]:
# ── Análisis del ruido: efecto sobre cada feature ────────────────────────────
np.random.seed(SEED)

# Seleccionar 5 ventanas representativas del conjunto de entrenamiento
sample_indices = [50, 200, 400, 700, 1000]
X_sample       = d['wins']['train'][sample_indices]   # (5, 30, 3)

noise_sigmas = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20]

fig, axes = plt.subplots(N_FEATURES, len(noise_sigmas),
                         figsize=(20, 8), sharey='row')

feat_labels = ['log_return', 'vol_21d', 'vol_zscore']
feat_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for col, sigma in enumerate(noise_sigmas):
    for row, feat_idx in enumerate(range(N_FEATURES)):
        # Usar la primera ventana de muestra
        original = X_sample[0, :, feat_idx]
        noisy    = original + np.random.normal(0, sigma, len(original))

        axes[row, col].plot(original, color=feat_colors[feat_idx],
                            linewidth=1.8, alpha=0.9, label='Original')
        if sigma > 0:
            axes[row, col].plot(noisy, color='#d62728', linewidth=0.9,
                                alpha=0.7, linestyle='--', label='Ruidosa')
            axes[row, col].fill_between(
                range(len(original)), original, noisy,
                alpha=0.15, color='#d62728'
            )
        snr = 20 * np.log10(original.std() / sigma) if sigma > 0 else np.inf
        snr_txt = f'SNR={snr:.0f}dB' if sigma > 0 else 'Sin ruido'

        if row == 0:
            title = f'σ={sigma}' + (' (selec.)' if sigma == NOISE_STDDEV else '')
            axes[row, col].set_title(title, fontweight='bold',
                                     color='green' if sigma == NOISE_STDDEV else 'black')
        if col == 0:
            axes[row, col].set_ylabel(feat_labels[feat_idx], fontsize=9)
        axes[row, col].text(0.05, 0.06, snr_txt,
                            transform=axes[row, col].transAxes,
                            fontsize=7, color='darkred')
        axes[row, col].tick_params(labelsize=7)

plt.suptitle('Efecto del Ruido Gaussiano sobre las Features (ventana de 30 días)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_gaussian_noise_features.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Dropout: Función Regularizadora en Series Temporales

### **8.1 Dropout en redes recurrentes**

El dropout estándar (Srivastava et al., 2014) silencia aleatoriamente una
fracción `p` de las neuronas en cada pasada hacia adelante durante el
entrenamiento, forzando al modelo a aprender representaciones redundantes
y robustas.

En redes recurrentes, el dropout se aplica en dos posibles ubicaciones:

**Dropout sobre la salida de la capa (implementado aquí):**
```
h'_t = Dropout(h_t, p)
```
Se aplica después de la capa LSTM/GRU, sobre la secuencia de estados ocultos.
La máscara de dropout se regenera en cada timestep (comportamiento por defecto
en Keras `layers.Dropout` después de una capa LSTM).

**Recurrent dropout (alternativa):**
```
h_t = LSTM(x_t, Dropout(h_{t-1}, p))
```
Aplica la misma máscara en todos los timesteps dentro de una secuencia,
que Keras implementa con el parámetro `recurrent_dropout`. Produce una
regularización más estable para secuencias largas pero puede ser
computacionalmente más costoso.

### **8.2 Efecto sobre la detección de anomalías**

El dropout cumple tres funciones en el contexto del DAE:

1. **Previene la memorización:** impide que neuronas específicas se
   especialicen en reconstruir secuencias individuales de entrenamiento.
2. **Amplía el margen de separación:** al introducir estocasticidad,
   el modelo aprende una distribución de reconstrucciones en lugar de una
   reconstrucción determinista, lo que aumenta la varianza del error de
   reconstrucción para datos fuera de distribución.
3. **Estimación de incertidumbre (MC Dropout):** si se mantiene activo
   durante la inferencia, el dropout permite estimar la incertidumbre de la
   reconstrucción mediante Monte Carlo sampling — una extensión útil para
   cuantificar la confianza en la alerta de anomalía.

In [ ]:
# ── MC Dropout: estimación de incertidumbre en la reconstrucción ──────────────
def mc_dropout_score(model_keras, X, n_samples=50):
    """
    Aplica Monte Carlo Dropout para estimar la incertidumbre
    del error de reconstrucción.

    Al llamar model(X, training=True) con el dropout activo,
    cada llamada produce una reconstrucción ligeramente diferente.
    La varianza entre llamadas estima la incertidumbre epistémica
    del modelo.

    Parámetros
    ----------
    model_keras : keras.Model con capas Dropout
    X           : np.ndarray (n, T, F) — ventanas de entrada
    n_samples   : int — número de muestras MC (más = más preciso, más lento)

    Retorna
    -------
    mse_mean : np.ndarray (n,) — MSE medio de reconstrucción
    mse_std  : np.ndarray (n,) — std del MSE (incertidumbre epistémica)
    """
    Xf = tf.constant(X.astype(np.float32))
    all_mse = []
    for _ in range(n_samples):
        # training=True activa el dropout durante la inferencia
        Xhat = model_keras(Xf, training=True).numpy()
        mse  = np.mean((X - Xhat) ** 2, axis=(1, 2))
        all_mse.append(mse)

    all_mse  = np.stack(all_mse, axis=0)  # (n_samples, n)
    return all_mse.mean(axis=0), all_mse.std(axis=0)


# Entrenamiento del DAE-TF para esta sección
print("Entrenando DAE-LSTM (TF) para el análisis de dropout...")
dae_tf.fit(d['wins']['train'], d['wins']['val'],
           batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
           patience_es=PATIENCE_ES, patience_lr=PATIENCE_LR)
dae_tf.set_threshold(d['wins']['train'], percentile=THRESHOLD_P := 95)
print("Listo.")

# Calcular MSE determinista y MC Dropout sobre el conjunto de val
X_val_np = d['wins']['val'].astype(np.float32)
mse_det  = dae_tf.score(X_val_np)
mse_mc_mean, mse_mc_std = mc_dropout_score(dae_tf.model, X_val_np, n_samples=30)

# Fechas de val
dates_val = d['dates']['val'][:len(mse_det)]

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# MSE determinista vs MC medio
axes[0].plot(dates_val, mse_det,     color='#1f77b4', linewidth=0.9,
             label='MSE determinista')
axes[0].plot(dates_val, mse_mc_mean, color='#d62728', linewidth=0.9,
             linestyle='--', alpha=0.85, label='MSE MC Dropout (media)')
axes[0].fill_between(dates_val,
                     mse_mc_mean - mse_mc_std,
                     mse_mc_mean + mse_mc_std,
                     alpha=0.20, color='#d62728',
                     label='±1 std (incertidumbre)')
axes[0].axhline(dae_tf.tau, color='green', linewidth=1.4,
                linestyle='--', label=f'τ (p{THRESHOLD_P})')
axes[0].set_title('MSE Determinista vs MC Dropout (Validación 2020)',
                  fontweight='bold')
axes[0].set_ylabel('MSE')
axes[0].legend(fontsize=8, loc='upper left')

# Incertidumbre (std) — zonas de alta incertidumbre
axes[1].fill_between(dates_val, 0, mse_mc_std,
                     color='#9467bd', alpha=0.6, label='Incertidumbre epistémica')
axes[1].set_title('Incertidumbre Epistémica (std del MSE bajo MC Dropout)',
                  fontweight='bold')
axes[1].set_ylabel('std(MSE)')
axes[1].legend(fontsize=9)

# Sombreado COVID
for ax in axes:
    ax.axvspan(pd.Timestamp('2020-02-15'), pd.Timestamp('2020-05-01'),
               alpha=0.10, color='red', label='COVID-19')

plt.suptitle(f'{NAMES[TICKER_PILOT]} — Análisis de Incertidumbre via MC Dropout',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_mc_dropout_uncertainty.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Error de Reconstrucción MSE: Definición y Propiedades

### **9.1 Definición formal**

El MSE de reconstrucción para la ventana que termina en el día t se define como:

```
score_t = MSE(x_t, x̂_t)
        = (1 / (T × F)) Σ_{τ=1}^{T} Σ_{f=1}^{F} (x_{τ,f} - x̂_{τ,f})²
        = ||x_t - x̂_t||²_F / (T × F)
```

donde:
- x_t ∈ R^{T×F} es la ventana de entrada limpia
- x̂_t ∈ R^{T×F} es la reconstrucción del decoder
- T = 30 (timesteps por ventana)
- F = 3 (features por timestep)
- ||·||_F es la norma de Frobenius

### **9.2 MSE por feature y por timestep**

Además del MSE global, se puede descomponer el error en:

**MSE por feature** (averaged over timesteps):
```
score_{t,f} = (1/T) Σ_{τ=1}^{T} (x_{τ,f} - x̂_{τ,f})²
```
Permite identificar qué feature fue más difícil de reconstruir.

**MSE por timestep** (averaged over features):
```
score_{t,τ} = (1/F) Σ_{f=1}^{F} (x_{τ,f} - x̂_{τ,f})²
```
Permite identificar en qué parte de la ventana ocurrió la mayor discrepancia.

### **9.3 Propiedades del MSE como score de anomalía**

1. **Sensibilidad cuadrática:** los errores grandes reciben peso cuadráticamente
   mayor. Una feature con error 10 contribuye 100 veces más al MSE que una con
   error 1. Esto amplifica la señal de anomalías severas.

2. **No negatividad:** MSE ≥ 0 siempre, con MSE = 0 sólo si x̂ = x.

3. **Escala dependiente de la normalización:** dado que las features están
   escaladas (μ=0, σ=1), el MSE tiene interpretación directa en unidades
   de desviaciones estándar al cuadrado.

4. **Distribución empírica sesgada:** la distribución del MSE en el conjunto
   de entrenamiento no es gaussiana (es sesgada a la derecha), por lo que
   el umbral τ se determina por el percentil empírico, no analíticamente.

In [ ]:
# ── Descomposición del MSE por feature y por timestep ────────────────────────
def compute_mse_decomposed(model_keras, X_windows):
    """
    Calcula el MSE descompuesto:
      - global   : (n,)       MSE promedio sobre T y F
      - by_feat  : (n, F)     MSE promedio sobre T, por feature
      - by_time  : (n, T)     MSE promedio sobre F, por timestep

    Parámetros
    ----------
    model_keras : keras.Model
    X_windows   : np.ndarray (n, T, F)
    """
    Xf    = X_windows.astype(np.float32)
    Xhat  = model_keras.predict(Xf, batch_size=256, verbose=0)
    sq_err= (Xf - Xhat) ** 2          # (n, T, F)

    mse_global  = sq_err.mean(axis=(1, 2))     # (n,)
    mse_by_feat = sq_err.mean(axis=1)          # (n, F)
    mse_by_time = sq_err.mean(axis=2)          # (n, T)

    return mse_global, mse_by_feat, mse_by_time, Xhat


mse_g, mse_f, mse_t, Xhat_val = compute_mse_decomposed(
    dae_tf.model, d['wins']['val']
)
dates_v = d['dates']['val'][:len(mse_g)]

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        'MSE Global (score de anomalía)',
        f'MSE por Feature  [{FEATURE_COLS[0]}, {FEATURE_COLS[1]}, {FEATURE_COLS[2]}]',
        'MSE por Timestep dentro de la ventana (media temporal)',
    ],
    vertical_spacing=0.07,
    row_heights=[0.35, 0.35, 0.30]
)

# MSE global
fig.add_trace(go.Scatter(x=dates_v, y=mse_g,
              mode='lines', name='MSE global',
              line=dict(color='#1f77b4', width=1.0)), row=1, col=1)
fig.add_hline(y=dae_tf.tau, line_dash='dash', line_color='red',
              annotation_text=f'τ={dae_tf.tau:.4f}',
              annotation_font_size=9, row=1, col=1)

# MSE por feature
feat_colors_plotly = ['#1f77b4', '#ff7f0e', '#2ca02c']
for fi, (fname, fc) in enumerate(zip(FEATURE_COLS, feat_colors_plotly)):
    fig.add_trace(go.Scatter(x=dates_v, y=mse_f[:, fi],
                  mode='lines', name=fname,
                  line=dict(color=fc, width=0.9)), row=2, col=1)

# MSE por timestep (promedio entre todas las ventanas — perfil de dificultad)
mse_t_mean = mse_t.mean(axis=0)   # (T,)
fig.add_trace(go.Bar(x=list(range(SEQ_LEN)), y=mse_t_mean,
              name='MSE medio por timestep',
              marker_color='#9467bd'), row=3, col=1)

# Bandas COVID
for ap in ANOMALY_PERIODS:
    for row in [1, 2]:
        fig.add_vrect(x0=ap['start'], x1=ap['end'],
                      fillcolor=ap['color'], layer='below',
                      line_width=0, row=row, col=1)

fig.update_layout(
    title_text=f'<b>{NAMES[TICKER_PILOT]} — Descomposición del MSE de Reconstrucción</b>',
    height=650, width=1100, template='plotly_white'
)
fig.show()


In [ ]:
# ── Análisis estadístico del MSE por split ────────────────────────────────────
print("PROPIEDADES ESTADÍSTICAS DEL MSE DE RECONSTRUCCIÓN")
print(f"Activo: {NAMES[TICKER_PILOT]} | Modelo: DAE-LSTM")
print("=" * 65)

for split in ['train', 'val', 'test']:
    mse_s = dae_tf.score(d['wins'][split])
    p5, p50, p95, p99 = (np.percentile(mse_s, q)
                          for q in [5, 50, 95, 99])
    print(f"\n{split.upper()} ({len(mse_s)} ventanas)")
    print(f"  Media    : {mse_s.mean():.6f}")
    print(f"  Std      : {mse_s.std():.6f}")
    print(f"  Asimetría: {stats.skew(mse_s):.4f}  "
          f"(> 0 confirma cola derecha pesada)")
    print(f"  Curtosis : {stats.kurtosis(mse_s):.4f}")
    print(f"  p5  : {p5:.6f}  |  p50 : {p50:.6f}  "
          f"|  p95 : {p95:.6f}  |  p99 : {p99:.6f}")
    jb_p = stats.jarque_bera(mse_s).pvalue
    print(f"  Jarque-Bera p-valor: {jb_p:.6f}  "
          f"({'rechaza normalidad' if jb_p < 0.05 else 'no rechaza'})")

print()
print(f"Umbral τ (p95 train): {dae_tf.tau:.6f}")


---
## Cómo el Modelo Aprende el Comportamiento Normal

### **10.1 El proceso de aprendizaje**

Durante el entrenamiento, el modelo procesa exclusivamente ventanas del
periodo 2015–2019. En cada época:

```
Para cada batch de ventanas {x₁, x₂, ..., x_B} del conjunto de train:

  1. Corrupción:    x̃ᵢ = xᵢ + εᵢ,  εᵢ ~ N(0, 0.05²)
  2. Codificación:  zᵢ = Encoder(x̃ᵢ)    ← representación latente
  3. Decodificación:x̂ᵢ = Decoder(zᵢ)   ← reconstrucción
  4. Pérdida:       L = mean((xᵢ - x̂ᵢ)²)  ← sobre entrada limpia
  5. Gradiente:     θ ← θ - α∇_θL          ← actualización Adam
```

### **10.2 Lo que el modelo aprende en cada componente**

**El encoder aprende:**
- Compresión de la dinámica temporal normal de los retornos.
- Correlaciones entre retorno, volatilidad y volumen en régimen normal.
- El horizonte de memoria relevante (efecto ARCH hasta ~30 días).
- Qué variaciones son ruido (ignorarlas) y qué son señal (preservarlas).

**El espacio latente codifica:**
- Cada dimensión z_k ∈ [-1,1] captura un modo de variación del mercado
  normal: tendencia de corto plazo, nivel de volatilidad, actividad inusual.
- En condiciones normales, z se concentra en una región compacta del
  hipercubo [-1,1]^16.

**El decoder aprende:**
- Cómo reconstruir la secuencia completa de 30 días a partir del vector z.
- Las dependencias temporales locales dentro de la ventana.
- La distribución condicional P(x_t | x_1,...,x_{t-1}) implícita en los datos normales.

In [ ]:
# ── Visualización del proceso de aprendizaje: curvas de pérdida ───────────────
hist = dae_tf.history
epochs    = range(1, len(hist.history['loss']) + 1)
train_loss= hist.history['loss']
val_loss  = hist.history['val_loss']
lr_hist   = hist.history.get('lr', [LR] * len(train_loss))
best_ep   = int(np.argmin(val_loss)) + 1

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Pérdida absoluta ─────────────────────────────────────────────────────────
axes[0].plot(epochs, train_loss, color='#1f77b4', linewidth=1.5,
             label='Train loss')
axes[0].plot(epochs, val_loss,   color='#d62728', linewidth=1.5,
             label='Val loss')
axes[0].axvline(best_ep, color='green', linewidth=1.4, linestyle='--',
                label=f'Mejor época: {best_ep}')
axes[0].set_title('Curvas de Pérdida (MSE)', fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MSE')
axes[0].legend()

# ── Pérdida en escala log ─────────────────────────────────────────────────────
axes[1].semilogy(epochs, train_loss, color='#1f77b4', linewidth=1.5,
                 label='Train loss')
axes[1].semilogy(epochs, val_loss,   color='#d62728', linewidth=1.5,
                 label='Val loss')
axes[1].axvline(best_ep, color='green', linewidth=1.4, linestyle='--',
                label=f'Mejor época: {best_ep}')
axes[1].set_title('Curvas de Pérdida (escala log)', fontweight='bold')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('log(MSE)')
axes[1].legend()

# ── Ratio val/train: monitoreo de sobreajuste ─────────────────────────────────
ratio = [v/t for v, t in zip(val_loss, train_loss)]
axes[2].plot(epochs, ratio, color='#9467bd', linewidth=1.5)
axes[2].axhline(1.0, color='black', linewidth=0.8, linestyle='--',
                label='ratio = 1.0 (sin gap)')
axes[2].axhline(1.5, color='orange', linewidth=0.8, linestyle=':',
                label='ratio = 1.5 (sobreajuste leve)')
axes[2].axvline(best_ep, color='green', linewidth=1.4, linestyle='--',
                label=f'Mejor época: {best_ep}')
axes[2].set_title('Ratio Val/Train (diagnóstico de sobreajuste)',
                  fontweight='bold')
axes[2].set_xlabel('Época')
axes[2].set_ylabel('Val Loss / Train Loss')
axes[2].legend(fontsize=8)

plt.suptitle(f'{NAMES[TICKER_PILOT]} — DAE-LSTM: Diagnóstico del Entrenamiento',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_training_diagnosis.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Diagnóstico de entrenamiento:")
print(f"  Train loss inicial  : {train_loss[0]:.6f}")
print(f"  Train loss final    : {train_loss[-1]:.6f}")
print(f"  Val   loss mínima   : {min(val_loss):.6f}")
print(f"  Ratio val/train     : {min(val_loss)/train_loss[best_ep-1]:.3f}")
print(f"  Reducción pérdida   : {(1 - train_loss[-1]/train_loss[0])*100:.1f}%")


In [ ]:
# ── Evolución del espacio latente durante el entrenamiento ─────────────────────
# Extraemos las representaciones latentes z y mostramos cómo
# los regímenes se separan en el espacio latente aprendido.

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# Obtener representaciones latentes sincronizando pesos
def sync_encoder_weights(dae_model, encoder_model):
    """Sincroniza los pesos del encoder desde el modelo completo."""
    for layer_name in ['enc_rnn_1', 'enc_drop_1', 'enc_rnn_2',
                       'enc_drop_2', 'latent_z']:
        try:
            src = dae_model.get_layer(layer_name)
            tgt = encoder_model.get_layer(layer_name)
            tgt.set_weights(src.get_weights())
        except Exception:
            pass

# Construir encoder de inferencia (sin GaussianNoise)
enc_inf = build_encoder_tf(SEQ_LEN, N_FEATURES, LATENT_DIM,
                            LSTM_UNITS, DROPOUT_RATE, 'LSTM')
sync_encoder_weights(dae_tf.model, enc_inf)

# Obtener representaciones latentes de train y val
z_train_arr = enc_inf.predict(d['wins']['train'].astype(np.float32),
                               batch_size=256, verbose=0)
z_val_arr   = enc_inf.predict(d['wins']['val'].astype(np.float32),
                               batch_size=256, verbose=0)

# Combinar y etiquetar
Z_all   = np.vstack([z_train_arr, z_val_arr])
y_regime= np.array([0]*len(z_train_arr) + [1]*len(z_val_arr))

# PCA 2D
pca    = PCA(n_components=2, random_state=SEED)
Z_pca  = pca.fit_transform(Z_all)

# t-SNE 2D
print("Calculando t-SNE del espacio latente...")
tsne   = TSNE(n_components=2, perplexity=40,
              n_iter=1000, random_state=SEED, learning_rate='auto')
Z_tsne = tsne.fit_transform(Z_all)
print("Listo.")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_lat = ['#1f77b4', '#d62728']
labels_lat = ['Normal (2015-2019)', 'Validación — COVID (2020)']

for proj, ax, title in [
    (Z_pca,  axes[0], 'PCA (2 componentes principales)'),
    (Z_tsne, axes[1], 't-SNE (perplexity=40)')
]:
    for regime in [0, 1]:
        mask = y_regime == regime
        ax.scatter(proj[mask, 0], proj[mask, 1],
                   c=colors_lat[regime], s=10 if regime == 0 else 20,
                   alpha=0.3 if regime == 0 else 0.7,
                   label=labels_lat[regime])
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
    ax.legend(fontsize=9)

plt.suptitle(
    f'{NAMES[TICKER_PILOT]} — Espacio Latente z ∈ R^{LATENT_DIM}: '
    f'PCA vs t-SNE',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('fig_latent_space_pca_tsne.png', dpi=120, bbox_inches='tight')
plt.show()

var_exp = pca.explained_variance_ratio_
print(f"PCA: varianza explicada por dim 1 = {var_exp[0]*100:.1f}%, "
      f"dim 2 = {var_exp[1]*100:.1f}%  (total = {sum(var_exp)*100:.1f}%)")


---
## Cómo se Detectan las Anomalías

### **11.1 El mecanismo de detección paso a paso**

```
PASO 1 — Construcción de la ventana
  Para cada día t en el conjunto de evaluación:
    ventana_t = [x_{t-29}, x_{t-28}, ..., x_t]  ∈ R^{30×3}
    (los últimos 30 días de features escaladas)

PASO 2 — Codificación (encoder)
  z_t = Encoder(ventana_t)  ∈ R^16
  El encoder produce una representación comprimida.
  Si ventana_t es similar a las vistas en entrenamiento:
    z_t cae en la región densa Z_normal del espacio latente.
  Si ventana_t es anómala:
    z_t cae en una región que el decoder nunca aprendió a decodificar bien.

PASO 3 — Decodificación (decoder)
  ventana_hat_t = Decoder(z_t)  ∈ R^{30×3}

PASO 4 — Score de anomalía
  score_t = MSE(ventana_t, ventana_hat_t)
          = ||ventana_t - ventana_hat_t||²_F / (30 × 3)

PASO 5 — Clasificación
  Si score_t > τ:  ANOMALÍA DETECTADA (alerta generada)
  Si score_t ≤ τ:  COMPORTAMIENTO NORMAL
```

### **11.2 Selección del umbral τ**

```
τ = percentil_p*(score_train)

donde p* = argmax_{p ∈ {85,90,93,95,97,99}} F1(validación, τ_p)
```

Este protocolo garantiza que τ sea la solución a:

```
minimize    FP_rate(τ)    (falsos positivos)
subject to  TP_rate(τ) ≥ objetivo_recall

equivalentemente: maximizar F1 = 2PR/(P+R)
```

### **11.3 Tipos de anomalías detectables**

El DAE puede detectar distintos tipos de anomalías financieras:

| Tipo | Descripción | Feature principal |
|---|---|---|
| **Retorno extremo** | |r_t| >> σ de entrenamiento | log_return |
| **Spike de volatilidad** | σ_t >> baseline de entrenamiento | vol_21d |
| **Volumen anómalo** | z_vol_t >> límite de entrenamiento | vol_zscore |
| **Cambio de correlación** | Combinación inusual de las 3 features | Multivariado |
| **Persistencia anómala** | Patrón secuencial sin precedente en train | Temporal (LSTM) |

In [ ]:
# ── Detección completa sobre todos los splits ─────────────────────────────────
scores = {s: dae_tf.score(d['wins'][s]) for s in ['train','val','test']}
preds  = {s: (scores[s] > dae_tf.tau).astype(int)
          for s in ['train','val','test']}

# Construir series temporales del score
def mse_series(split):
    s = scores[split][:len(d['dates'][split])]
    return pd.Series(s, index=d['dates'][split])

mse_full = pd.concat([mse_series('train'),
                       mse_series('val'),
                       mse_series('test')]).sort_index()

# ── Visualización principal ───────────────────────────────────────────────────
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[
        f'{NAMES[TICKER_PILOT]} — Score de Anomalía MSE (2015-2024)',
        f'{NAMES[TICKER_PILOT]} — Alertas de Anomalía Generadas',
        f'{NAMES[TICKER_PILOT]} — Precio de Cierre con Alertas',
    ],
    vertical_spacing=0.06,
    row_heights=[0.45, 0.25, 0.30]
)

# Score MSE
fig.add_trace(go.Scatter(x=mse_full.index, y=mse_full.values,
              mode='lines', name='MSE',
              line=dict(color='#1f77b4', width=0.9)), row=1, col=1)
fig.add_hline(y=dae_tf.tau, line_dash='dash', line_color='red',
              annotation_text=f'τ = {dae_tf.tau:.4f}',
              annotation_font_size=9, row=1, col=1)

# Señal binaria de alerta
alert_dates  = mse_full[mse_full > dae_tf.tau].index
alert_scores = mse_full[mse_full > dae_tf.tau].values
fig.add_trace(go.Scatter(x=alert_dates, y=alert_scores,
              mode='markers', name='Anomalía detectada',
              marker=dict(color='#d62728', size=4, symbol='circle')),
              row=1, col=1)

alert_binary = (mse_full > dae_tf.tau).astype(int)
fig.add_trace(go.Scatter(x=alert_binary.index, y=alert_binary.values,
              mode='lines', name='Alerta (0/1)',
              line=dict(color='#d62728', width=1.2),
              fill='tozeroy', fillcolor='rgba(214,39,40,0.15)'),
              row=2, col=1)

# Precio de cierre (desde raw data)
price_raw = build_features(TICKER_PILOT)
price_close = yf.download(TICKER_PILOT, start=TRAIN_START, end=TEST_END,
                           auto_adjust=True, progress=False)['Close']
fig.add_trace(go.Scatter(x=price_close.index, y=price_close.values,
              mode='lines', name='Precio cierre',
              line=dict(color='#7f7f7f', width=1.0)),
              row=3, col=1)
fig.add_trace(go.Scatter(x=alert_dates,
              y=price_close.reindex(alert_dates).values,
              mode='markers', name='Alerta',
              marker=dict(color='#d62728', size=4)),
              row=3, col=1)

# Bandas de anomalía
for ap in ANOMALY_PERIODS:
    for row in [1, 2, 3]:
        fig.add_vrect(x0=ap['start'], x1=ap['end'],
                      fillcolor=ap['color'], layer='below',
                      line_width=0,
                      annotation_text=ap['name'] if row == 1 else '',
                      annotation_font_size=9,
                      row=row, col=1)

# Líneas de split
for split_d in [TRAIN_END, VAL_END]:
    for row in [1, 2, 3]:
        fig.add_vline(x=split_d, line_dash='dot',
                      line_color='black', line_width=0.7,
                      row=row, col=1)

fig.update_layout(
    height=700, width=1150, template='plotly_white',
    title_text=f'<b>Detección de Anomalías Completa — {NAMES[TICKER_PILOT]} '
               f'DAE-LSTM (2015-2024)</b>'
)
fig.show()


---
## Visualización Profunda del Error de Reconstrucción

### **Motivación**

Más allá del score escalar por ventana, es informativo visualizar:
1. El mapa de calor del error de reconstrucción (tiempo × feature).
2. Las ventanas más anómalas vs. más normales (comparación directa).
3. La evolución del error a lo largo de la ventana temporal.

In [ ]:
# ── Mapa de calor: error de reconstrucción por fecha × feature ──────────────
# Calcular error por fecha y por feature sobre el conjunto de validación
mse_g_v, mse_f_v, mse_t_v, Xhat_v = compute_mse_decomposed(
    dae_tf.model, d['wins']['val']
)
dates_v_plot = d['dates']['val'][:len(mse_g_v)]

# Crear DataFrame de error por feature indexado por fecha
err_feat_df = pd.DataFrame(
    mse_f_v,
    index=dates_v_plot,
    columns=FEATURE_COLS
)

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1, 1, 1]})

# MSE global
axes[0].fill_between(err_feat_df.index, mse_g_v,
                     color='#1f77b4', alpha=0.5)
axes[0].plot(err_feat_df.index, mse_g_v,
             color='#1f77b4', linewidth=0.8)
axes[0].axhline(dae_tf.tau, color='red', linewidth=1.4,
                linestyle='--', label=f'τ = {dae_tf.tau:.4f}')
axes[0].axvspan(pd.Timestamp('2020-02-15'), pd.Timestamp('2020-05-01'),
                alpha=0.12, color='red')
axes[0].set_title('MSE Global de Reconstrucción', fontweight='bold')
axes[0].set_ylabel('MSE')
axes[0].legend(fontsize=9)

feat_colors2 = ['#1f77b4', '#ff7f0e', '#2ca02c']
for i, (feat, fc) in enumerate(zip(FEATURE_COLS, feat_colors2)):
    axes[i+1].fill_between(err_feat_df.index,
                            err_feat_df[feat], color=fc, alpha=0.5)
    axes[i+1].plot(err_feat_df.index, err_feat_df[feat],
                   color=fc, linewidth=0.8)
    axes[i+1].axvspan(pd.Timestamp('2020-02-15'), pd.Timestamp('2020-05-01'),
                      alpha=0.12, color='red')
    axes[i+1].set_title(f'MSE — {feat}', fontweight='bold')
    axes[i+1].set_ylabel('MSE')

axes[-1].set_xlabel('Fecha')
plt.suptitle(
    f'{NAMES[TICKER_PILOT]} — Error de Reconstrucción por Feature (Validación 2020)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('fig_mse_by_feature_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Comparación: Top 3 ventanas más anómalas vs Top 3 más normales ────────────
n_show  = 3
top_anom = np.argsort(mse_g_v)[-n_show:][::-1]    # mayor MSE
top_norm = np.argsort(mse_g_v)[:n_show]             # menor MSE

fig, axes = plt.subplots(N_FEATURES, n_show * 2,
                         figsize=(20, 9), sharey='row')

feat_labels2 = ['log_return (r_t)', 'vol_21d (σ_t)', 'vol_zscore']

for col_group, (indices, title_prefix, color_orig, color_rec) in enumerate([
    (top_norm, 'NORMAL',  '#1f77b4', '#aec7e8'),
    (top_anom, 'ANÓMALA', '#d62728', '#ff9896'),
]):
    for j, win_idx in enumerate(indices):
        col_abs = col_group * n_show + j
        x_orig  = d['wins']['val'][win_idx]             # (30, 3)
        x_recon = Xhat_v[win_idx]                       # (30, 3)
        mse_val = mse_g_v[win_idx]
        win_date= str(dates_v_plot[win_idx].date())

        for row_idx in range(N_FEATURES):
            axes[row_idx, col_abs].plot(x_orig[:, row_idx],
                                        color=color_orig,
                                        linewidth=1.8, label='Original')
            axes[row_idx, col_abs].plot(x_recon[:, row_idx],
                                        color=color_rec, linewidth=1.2,
                                        linestyle='--', alpha=0.9,
                                        label='Reconstrucción')
            axes[row_idx, col_abs].fill_between(
                range(SEQ_LEN),
                x_orig[:, row_idx], x_recon[:, row_idx],
                alpha=0.20, color=color_rec
            )
            if row_idx == 0:
                axes[row_idx, col_abs].set_title(
                    f'{title_prefix}
{win_date}
MSE={mse_val:.5f}',
                    fontweight='bold', fontsize=9,
                    color='darkred' if title_prefix == 'ANÓMALA' else 'navy'
                )
            if col_abs == 0:
                axes[row_idx, col_abs].set_ylabel(feat_labels2[row_idx],
                                                   fontsize=8)
            if row_idx == 0 and col_abs == 0:
                axes[row_idx, col_abs].legend(fontsize=7, loc='upper right')

plt.suptitle(
    f'{NAMES[TICKER_PILOT]} — Top 3 Ventanas Normales (izq.) vs Anómalas (der.)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('fig_normal_vs_anomalous_windows.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Análisis por Feature: ¿Qué Reconstruye Mal durante Anomalías?

### **Motivación**

Identificar qué feature contribuye más al error de reconstrucción durante
un periodo de crisis proporciona interpretabilidad al modelo: no sólo
detecta una anomalía, sino que indica su naturaleza (retorno extremo,
volatilidad inusual, actividad de mercado atípica).

In [ ]:
# ── Contribución de cada feature al MSE total por periodo ────────────────────
def feature_mse_contribution(wins_arr, model_k, feature_cols):
    """
    Calcula la contribución porcentual de cada feature al MSE total.
    Retorna DataFrame con columnas = features, filas = ventanas.
    """
    Xf    = wins_arr.astype(np.float32)
    Xhat  = model_k.predict(Xf, batch_size=256, verbose=0)
    sq_e  = (Xf - Xhat) ** 2         # (n, T, F)
    per_f = sq_e.mean(axis=1)         # (n, F) — media sobre T
    total = per_f.sum(axis=1, keepdims=True)  # (n, 1)
    pct   = per_f / (total + 1e-10) * 100     # (n, F) porcentajes
    return pct   # (n, F)


# Calcular sobre validación
contrib_val = feature_mse_contribution(
    d['wins']['val'], dae_tf.model, FEATURE_COLS
)
dates_v2    = d['dates']['val'][:len(contrib_val)]

# Separar periodos
idx_normal  = (pd.Series(d['y']['val'][:len(dates_v2)],
                          index=dates_v2) == 0)
idx_covid   = (pd.Series(d['y']['val'][:len(dates_v2)],
                          index=dates_v2) == 1)

contrib_normal = contrib_val[idx_normal.values]
contrib_covid  = contrib_val[idx_covid.values]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for j, (contrib, title, color) in enumerate([
    (contrib_normal, 'Régimen Normal (2020, no-COVID)', '#1f77b4'),
    (contrib_covid,  'Régimen COVID-19 (Feb-May 2020)', '#d62728'),
    (contrib_val,    'Validación Completa (2020)',       '#9467bd'),
]):
    means = contrib.mean(axis=0)
    stds  = contrib.std(axis=0)

    bars = axes[j].bar(FEATURE_COLS, means, color=color,
                       alpha=0.75, edgecolor='white')
    axes[j].errorbar(FEATURE_COLS, means, yerr=stds,
                     fmt='none', color='black', capsize=5, linewidth=1.2)

    for bar, val in zip(bars, means):
        axes[j].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.5,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=10)

    axes[j].set_title(title, fontweight='bold')
    axes[j].set_ylabel('Contribución al MSE (%)')
    axes[j].set_ylim(0, 65)
    axes[j].tick_params(axis='x', rotation=15)

plt.suptitle(
    f'{NAMES[TICKER_PILOT]} — Contribución de Cada Feature al Error de Reconstrucción',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('fig_feature_mse_contribution.png', dpi=120, bbox_inches='tight')
plt.show()

print("Contribución media al MSE por régimen:")
print(f"{'Feature':<15}  {'Normal (%)':>12}  {'COVID (%)':>12}  {'Delta':>10}")
print('-' * 52)
for i, feat in enumerate(FEATURE_COLS):
    n_m = contrib_normal[:, i].mean()
    c_m = contrib_covid[:, i].mean()
    print(f"{feat:<15}  {n_m:>12.2f}  {c_m:>12.2f}  {c_m-n_m:>+10.2f}")


In [ ]:
# ── Evaluación final del DAE-LSTM y DAE-GRU ──────────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

def eval_dae(cell_type):
    dae = DenoisingAutoencoderTF(
        SEQ_LEN, N_FEATURES, LATENT_DIM, LSTM_UNITS,
        DROPOUT_RATE, NOISE_STDDEV, cell_type
    )
    dae.fit(d['wins']['train'], d['wins']['val'],
            BATCH_SIZE, MAX_EPOCHS, PATIENCE_ES, PATIENCE_LR)
    dae.set_threshold(d['wins']['train'], 95)

    sc_test = dae.score(d['wins']['test'])
    y_true  = d['y']['test'][:len(sc_test)]
    y_pred  = (sc_test > dae.tau).astype(int)

    return {
        'Modelo':    f'DAE-{cell_type}',
        'Precisión': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_true,    y_pred, zero_division=0), 4),
        'F1':        round(f1_score(y_true,        y_pred, zero_division=0), 4),
        'AUC-ROC':   round(roc_auc_score(y_true, sc_test), 4),
    }

print("EVALUACIÓN FINAL — TEST SET (2021-2024)")
print("=" * 55)
for ct in ['LSTM', 'GRU']:
    res = eval_dae(ct)
    print(f"  {res['Modelo']:<12}  P={res['Precisión']:.4f}  "
          f"R={res['Recall']:.4f}  F1={res['F1']:.4f}  "
          f"AUC={res['AUC-ROC']:.4f}")


---
## Resumen del Notebook

### **Implementaciones entregadas**

| Componente | TensorFlow/Keras | PyTorch |
|---|---|---|
| `GaussianNoiseLayer` | Capa personalizada, integrada al grafo | Loop de entrenamiento explícito |
| Encoder LSTM/GRU | `build_encoder_tf()` — sub-modelo reutilizable | `DAE_PyTorch.encode()` |
| Decoder | `RepeatVector + LSTM/GRU + TimeDistributed` | `repeat + GRU/LSTM + Linear` |
| Wrapper | `DenoisingAutoencoderTF` | `DenoisingAutoencoderPT` |
| MC Dropout | `mc_dropout_score()` | Activar `model.train()` en inferencia |

### **Cadena lógica del modelo**

```
DATOS NORMALES (train 2015-2019)
        │
        ▼
CORRUPCIÓN: x̃ = x + ε  (ε ~ N(0, 0.05²))
        │
        ▼
ENCODER: z = LSTM(64) → LSTM(32) → Dense(16)  [compresión 5.6×]
        │           ↑
        │    aprende la "gramática" del mercado normal
        ▼
ESPACIO LATENTE z ∈ R^16 — región compacta para datos normales
        │
        ▼
DECODER: RepeatVector → LSTM(32) → LSTM(64) → Dense(3)
        │
        ▼
x̂ = RECONSTRUCCIÓN  →  LOSS = MSE(x, x̂)  →  BACKPROP  →  θ actualizado
        │
        ▼ (inferencia sobre cualquier split)
SCORE: MSE(x, Decoder(Encoder(x)))
        │
        ├── score ≤ τ  →  NORMAL
        └── score > τ  →  ANOMALÍA DETECTADA
```

**Próximo paso:** `Notebook_06_Analisis_Final.ipynb` — análisis profundo
de los resultados, comparación con benchmarks y reporte ejecutivo.